# Train Colenet on the Uploadable Cholec80-CSV 5 FPS Dataset

This notebook is a standalone version of the repository training and testing pipeline. It embeds the dataset, model, training, validation, testing, checkpointing, and logging code in one place.

Expected dataset root: `data/cholec80_csv_uploadable_5fps/`.

## 1. Setup and Config

Run all cells from top to bottom. The notebook creates a fresh timestamped folder under `log/` for each run.

In [ ]:
import copy
import json
import os
import random
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from skimage import io
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm


def find_project_root(start):
    for path in [start, *start.parents]:
        if (path / "data").exists():
            return path
    raise FileNotFoundError("Could not find a project root containing a data/ directory.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATASET_ROOT = PROJECT_ROOT / "data" / "cholec80_csv_uploadable_5fps"
LOG_ROOT = PROJECT_ROOT / "log"

BACKBONE = "resnet"
EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 1e-5
TARGET_METRIC = "mean_f1"
THRESHOLD = 0.5
SEED = 990411
NUM_WORKERS = 4

LABEL_COLUMNS = ["two_structures_score", "cystic_plate_score", "hc_triangle_score"]
CLASS_NAMES = ["two_structures", "cystic_plate", "hc_triangle"]
DEFAULT_POS_WEIGHT = [1.4380261927034612, 6.650975047179703, 2.993378570646821]

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_NAME = f"notebook_{BACKBONE}_{RUN_TIMESTAMP}"
RUN_DIR = LOG_ROOT / RUN_NAME

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"


def set_random_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)


set_random_seed(SEED)

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)
print("Run directory:", RUN_DIR)
print("Device:", DEVICE)

## 2. Dataset Preflight and Inspection

This cell fails early if the uploadable dataset has not been created yet.

In [ ]:
SPLIT_INFO = {
    "train": DATASET_ROOT / "train" / "annotations.csv",
    "valid": DATASET_ROOT / "valid" / "annotations.csv",
    "test": DATASET_ROOT / "test" / "annotations.csv",
}

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"Uploadable dataset folder does not exist: {DATASET_ROOT}\n"
        "Run notebooks/create_uploadable_dataset.ipynb first."
    )

missing_files = [str(path) for path in SPLIT_INFO.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing required annotation files:\n" + "\n".join(missing_files) +
        "\nRun notebooks/create_uploadable_dataset.ipynb first."
    )


def load_split(split_name, annotations_path):
    df = pd.read_csv(annotations_path)
    required_columns = {"video_name", "image", "relative_path", *LABEL_COLUMNS}
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise ValueError(f"{annotations_path} is missing columns: {sorted(missing_columns)}")
    return df


def validate_frame_paths(split_name, df):
    split_dir = DATASET_ROOT / split_name
    missing = [split_dir / rel_path for rel_path in df["relative_path"] if not (split_dir / rel_path).exists()]
    if missing:
        examples = "\n".join(str(path) for path in missing[:10])
        raise FileNotFoundError(f"{split_name} has {len(missing)} missing frames. Examples:\n{examples}")


def split_summary(split_name, df):
    binary_labels = df[LABEL_COLUMNS].clip(upper=1)
    summary = {
        "rows": int(len(df)),
        "unique_frames": int(df[["video_name", "image"]].drop_duplicates().shape[0]),
        "videos": int(df["video_name"].nunique()),
    }
    for column in LABEL_COLUMNS:
        positives = int(binary_labels[column].sum())
        negatives = int(len(binary_labels) - positives)
        summary[f"{column}_positive"] = positives
        summary[f"{column}_negative"] = negatives
        summary[f"{column}_pos_weight"] = round(negatives / positives, 6) if positives else None
    return summary


split_dfs = {}
summaries = {}
for split_name, annotations_path in SPLIT_INFO.items():
    df = load_split(split_name, annotations_path)
    validate_frame_paths(split_name, df)
    split_dfs[split_name] = df
    summaries[split_name] = split_summary(split_name, df)

print(json.dumps(summaries, indent=2))
print("Dataset preflight passed.")

## 3. Dataset, Model, and Metric Utilities

These are embedded notebook versions of the repository classes, adjusted to read from the uploadable subset structure.

In [ ]:
class Cholec80UploadableDataset(Dataset):
    def __init__(self, split_name, dataset_root, annotations_df, transform=None):
        assert split_name in ["train", "valid", "test"]
        self.split_name = split_name
        self.split_dir = Path(dataset_root) / split_name
        self.labels_df = annotations_df.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, index):
        row = self.labels_df.iloc[index]
        relative_path = row["relative_path"]
        image_path = self.split_dir / relative_path
        image = io.imread(str(image_path))
        if image.ndim == 2:
            image = np.stack([image, image, image], axis=-1)
        image = image[:, :, :3]

        target = np.array([int(min(1, row[column])) for column in LABEL_COLUMNS], dtype=np.float32)
        if self.transform:
            image = self.transform(image)
        return image, target, f"{self.split_name}/{relative_path}"


class ColeNet(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.model = self.get_backbone(backbone)

    def _weights_or_pretrained(self, constructor, weights_class=None, **kwargs):
        if weights_class is not None:
            try:
                return constructor(weights=weights_class.DEFAULT, **kwargs)
            except Exception as exc:
                warnings.warn(f"Falling back to pretrained=True because weights API failed: {exc}")
        return constructor(pretrained=True, **kwargs)

    def get_backbone(self, backbone):
        if backbone == "vgg":
            weights = getattr(models, "VGG16_Weights", None)
            model = self._weights_or_pretrained(models.vgg16, weights)
            model.classifier = nn.Linear(model.classifier[0].in_features, 3)
        elif backbone == "resnet":
            weights = getattr(models, "ResNet50_Weights", None)
            model = self._weights_or_pretrained(models.resnet50, weights)
            model.fc = nn.Linear(model.fc.in_features, 3)
        elif backbone == "alexnet":
            weights = getattr(models, "AlexNet_Weights", None)
            model = self._weights_or_pretrained(models.alexnet, weights)
            model.classifier = nn.Linear(model.classifier[1].in_features, 3)
        elif backbone == "densenet":
            weights = getattr(models, "DenseNet169_Weights", None)
            model = self._weights_or_pretrained(models.densenet169, weights)
            model.classifier = nn.Linear(model.classifier.in_features, 3)
        elif backbone == "inception":
            weights = getattr(models, "Inception_V3_Weights", None)
            model = self._weights_or_pretrained(models.inception_v3, weights)
            model.fc = nn.Linear(model.fc.in_features, 3)
        else:
            raise NotImplementedError(f"Unknown model backbone: {backbone}")
        return model

    def forward(self, x):
        outputs = self.model(x)
        if isinstance(outputs, tuple):
            outputs = outputs[0]
        return outputs


def compute_metrics(labels, probabilities, losses, threshold=0.5):
    y = np.vstack(labels).astype(np.float32)
    y_prob = np.vstack(probabilities).astype(np.float32)
    y_hat = (y_prob >= threshold).astype(np.float32)

    metric_values = {}
    accuracies = []
    precisions = []
    recalls = []
    f1s = []

    for index, name in enumerate(CLASS_NAMES):
        accuracy = accuracy_score(y[:, index], y_hat[:, index])
        precision = precision_score(y[:, index], y_hat[:, index], zero_division=0)
        recall = recall_score(y[:, index], y_hat[:, index], zero_division=0)
        f1 = f1_score(y[:, index], y_hat[:, index], zero_division=0)

        metric_values[f"{name}_accuracy"] = accuracy
        metric_values[f"{name}_precision"] = precision
        metric_values[f"{name}_recall"] = recall
        metric_values[f"{name}_f1"] = f1
        accuracies.append(accuracy)
        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)

    metric_values["mean_acc"] = float(np.mean(accuracies))
    metric_values["mean_precision"] = float(np.mean(precisions))
    metric_values["mean_recall"] = float(np.mean(recalls))
    metric_values["mean_f1"] = float(np.mean(f1s))
    metric_values["loss"] = float(np.sum(losses))
    metric_values["n_samples"] = int(y.shape[0])
    return metric_values


def metrics_to_row(epoch, split_name, metrics):
    row = {"epoch": epoch, "split": split_name}
    row.update(metrics)
    return row


def write_metrics_csv(path, rows):
    pd.DataFrame(rows).to_csv(path, index=False)

## 4. Build Datasets and DataLoaders

In [ ]:
data_normalization = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    data_normalization,
])
eval_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    data_normalization,
])

train_dataset = Cholec80UploadableDataset("train", DATASET_ROOT, split_dfs["train"], transform=train_transforms)
valid_dataset = Cholec80UploadableDataset("valid", DATASET_ROOT, split_dfs["valid"], transform=eval_transforms)
test_dataset = Cholec80UploadableDataset("test", DATASET_ROOT, split_dfs["test"], transform=eval_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print("Train batches:", len(train_loader))
print("Valid batches:", len(valid_loader))
print("Test batches:", len(test_loader))

## 5. Training Helpers and Run Directory

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, split_name="train", epoch=0):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    labels_list = []
    probabilities_list = []
    losses = []
    prediction_rows = []

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for inputs, labels, paths in tqdm(loader, total=len(loader), desc=f"{split_name} epoch {epoch}"):
            inputs = inputs.to(DEVICE, non_blocking=PIN_MEMORY).float()
            labels = labels.to(DEVICE, non_blocking=PIN_MEMORY).float()

            if is_train:
                optimizer.zero_grad()

            logits = model(inputs)
            loss = criterion(logits, labels)

            if is_train:
                loss.backward()
                optimizer.step()

            probabilities = torch.sigmoid(logits)
            labels_np = labels.detach().cpu().numpy()
            probabilities_np = probabilities.detach().cpu().numpy()

            labels_list.append(labels_np)
            probabilities_list.append(probabilities_np)
            losses.append(loss.item())

            if split_name == "test":
                binary_np = (probabilities_np >= THRESHOLD).astype(int)
                for i, path in enumerate(paths):
                    prediction_rows.append({
                        "path": path,
                        "true_two_structures_score": int(labels_np[i, 0]),
                        "true_cystic_plate_score": int(labels_np[i, 1]),
                        "true_hc_triangle_score": int(labels_np[i, 2]),
                        "prob_two_structures_score": float(probabilities_np[i, 0]),
                        "prob_cystic_plate_score": float(probabilities_np[i, 1]),
                        "prob_hc_triangle_score": float(probabilities_np[i, 2]),
                        "pred_two_structures_score": int(binary_np[i, 0]),
                        "pred_cystic_plate_score": int(binary_np[i, 1]),
                        "pred_hc_triangle_score": int(binary_np[i, 2]),
                    })

    metrics = compute_metrics(labels_list, probabilities_list, losses, threshold=THRESHOLD)
    return metrics, prediction_rows


LOG_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=False)

config = {
    "project_root": str(PROJECT_ROOT),
    "dataset_root": str(DATASET_ROOT),
    "run_name": RUN_NAME,
    "backbone": BACKBONE,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "target_metric": TARGET_METRIC,
    "threshold": THRESHOLD,
    "seed": SEED,
    "num_workers": NUM_WORKERS,
    "device": str(DEVICE),
    "pos_weight": DEFAULT_POS_WEIGHT,
    "split_summary": summaries,
}
with open(RUN_DIR / "config.json", "w") as handle:
    json.dump(config, handle, indent=2)

print("Created run directory:", RUN_DIR)

## 6. Train and Validate

This cell performs the full training run and saves `best_model.pth` whenever validation `mean_f1` improves.

In [ ]:
model = ColeNet(BACKBONE).to(DEVICE)
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9, weight_decay=0.005)
pos_weight = torch.tensor(DEFAULT_POS_WEIGHT, dtype=torch.float32, device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

train_log_rows = []
valid_log_rows = []
epoch_summary_rows = []
best_score = -float("inf")
best_epoch = None
best_model_state = None

for epoch in range(1, EPOCHS + 1):
    print(f"Epoch {epoch} of {EPOCHS}")
    train_metrics, _ = run_epoch(model, train_loader, criterion, optimizer=optimizer, split_name="train", epoch=epoch)
    valid_metrics, _ = run_epoch(model, valid_loader, criterion, optimizer=None, split_name="valid", epoch=epoch)

    train_row = metrics_to_row(epoch, "train", train_metrics)
    valid_row = metrics_to_row(epoch, "valid", valid_metrics)
    train_log_rows.append(train_row)
    valid_log_rows.append(valid_row)

    write_metrics_csv(RUN_DIR / "train_log.csv", train_log_rows)
    write_metrics_csv(RUN_DIR / "valid_log.csv", valid_log_rows)

    current_score = valid_metrics[TARGET_METRIC]
    is_best = current_score > best_score
    if is_best:
        best_score = current_score
        best_epoch = epoch
        best_model_state = copy.deepcopy(model.state_dict())
        torch.save(best_model_state, RUN_DIR / "best_model.pth")
        print(f"New best {TARGET_METRIC}: {best_score:.6f}")

    epoch_summary_rows.append({
        "epoch": epoch,
        "train_mean_f1": train_metrics["mean_f1"],
        "valid_mean_f1": valid_metrics["mean_f1"],
        "train_loss": train_metrics["loss"],
        "valid_loss": valid_metrics["loss"],
        "is_best": is_best,
        "best_epoch": best_epoch,
        "best_score": best_score,
    })
    pd.DataFrame(epoch_summary_rows).to_csv(RUN_DIR / "epoch_summary.csv", index=False)

torch.save(model.state_dict(), RUN_DIR / "last_model.pth")
print("Training complete.")
print("Best epoch:", best_epoch)
print(f"Best {TARGET_METRIC}: {best_score:.6f}")

## 7. Test Best Model

This loads `best_model.pth`, evaluates on the test split, and writes `test_log.csv` plus `test_predictions.csv`.

In [ ]:
best_model_path = RUN_DIR / "best_model.pth"
if not best_model_path.exists():
    raise FileNotFoundError(f"Missing best model checkpoint: {best_model_path}")

test_model = ColeNet(BACKBONE).to(DEVICE)
test_model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
test_model.eval()

test_metrics, test_prediction_rows = run_epoch(test_model, test_loader, criterion, optimizer=None, split_name="test", epoch=best_epoch)
test_log_rows = [metrics_to_row(best_epoch, "test", test_metrics)]
write_metrics_csv(RUN_DIR / "test_log.csv", test_log_rows)
pd.DataFrame(test_prediction_rows).to_csv(RUN_DIR / "test_predictions.csv", index=False)

print("Test metrics:")
print(json.dumps(test_metrics, indent=2))
print("Wrote predictions:", RUN_DIR / "test_predictions.csv")

## 8. Final Verification

In [ ]:
required_outputs = [
    RUN_DIR / "best_model.pth",
    RUN_DIR / "last_model.pth",
    RUN_DIR / "config.json",
    RUN_DIR / "train_log.csv",
    RUN_DIR / "valid_log.csv",
    RUN_DIR / "test_log.csv",
    RUN_DIR / "epoch_summary.csv",
    RUN_DIR / "test_predictions.csv",
]
missing_outputs = [str(path) for path in required_outputs if not path.exists()]
if missing_outputs:
    raise FileNotFoundError("Missing expected outputs:\n" + "\n".join(missing_outputs))

test_predictions = pd.read_csv(RUN_DIR / "test_predictions.csv")
if len(test_predictions) != len(split_dfs["test"]):
    raise AssertionError(
        f"Expected {len(split_dfs['test'])} test predictions, found {len(test_predictions)}"
    )

print("Verification passed.")
print("Run outputs are in:", RUN_DIR)